# scicp — Fine-tune MiniLM on Scripture (English)

Fine-tunes `all-MiniLM-L6-v2` on **~461k** training pairs from 9 English sources:
- LDS ↔ YLT translation pairs (KJV-style ↔ hyper-literal paraphrase)
- LDS ↔ Rotherham's Emphasized Bible translation pairs (1902, public domain)
- Topical guide (topic name ↔ verse text)
- Triple Combination Index (topic name ↔ verse text)
- Cross-references (theologically linked verses)
- kNN neighbors (semantically similar verses)
- Adjacent verses (chapter continuity)
- Same-topic verse pairs (TG + Triple Index)
- Strong's Hebrew/Greek semantic expansion pairs (14k lexicon entries)

**Platform: Kaggle (T4 GPU) — ~20–25 minutes.**

**Steps:**
1. Upload `resources/training-pairs.json` as a private Kaggle Dataset
2. Add that dataset to this notebook (it will appear at `/kaggle/input/<dataset-slug>/`)

3. Run all cells top to bottom4. Download the output model from the Kaggle session's Output tab

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {gpu}  VRAM: {vram} GB")
    # Auto-select batch size — L6 is smaller, fits more per batch
    if vram >= 40:    MICRO_BATCH, GRAD_ACCUM = 512, 1   # A100
    elif vram >= 22:  MICRO_BATCH, GRAD_ACCUM = 256, 1   # L4/A10
    else:             MICRO_BATCH, GRAD_ACCUM = 128, 2   # T4 (15.6 GB)
    print(f"→ micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}, effective_batch={MICRO_BATCH * GRAD_ACCUM}")
else:
    MICRO_BATCH, GRAD_ACCUM = 32, 8
    print("No GPU — training will be very slow")

In [ ]:
# ── 3. Locate training data ──────────────────────────────────────────────────
# On Kaggle: add the dataset via Add Data → Your Datasets → training-pairs
# It will be available at /kaggle/input/<dataset-slug>/training-pairs.json
import glob, os, time

# Auto-find training-pairs.json under /kaggle/input/
candidates = glob.glob("/kaggle/input/**/training-pairs.json", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "training-pairs.json not found under /kaggle/input/. "
        "Add the dataset via Add Data in the Kaggle notebook UI."
    )
LOCAL_PATH = candidates[0]
print(f"Found: {LOCAL_PATH}  ({os.path.getsize(LOCAL_PATH)//1024//1024} MB)")

In [ ]:
# ── 4. Load + prepare dataset ────────────────────────────────────────────────
import json, random
from datasets import Dataset

t0 = time.time()
with open(LOCAL_PATH) as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs):,} pairs in {time.time()-t0:.1f}s")

random.seed(42)
random.shuffle(pairs)

# 97/3 split — with 676k pairs, 3% validation (~20k) is more than enough
split = int(len(pairs) * 0.97)
train_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[:split]],
    "positive": [p["positive"] for p in pairs[:split]],
})
val_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[split:]],
    "positive": [p["positive"] for p in pairs[split:]],
})
del pairs  # free ~400 MB RAM

print(f"train={len(train_ds):,}  val={len(val_ds):,}")
print("Sample:", train_ds[0])

In [ ]:
# ── 5. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

# English-only model: 6 layers, 384 dims, fast inference
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OUT_DIR    = "/content/scripture-minilm"
EPOCHS     = 2       # 346k × 2 = 692k samples; plenty for convergence

effective_batch = MICRO_BATCH * GRAD_ACCUM  # 256 on T4
total_steps     = (len(train_ds) // effective_batch) * EPOCHS
warmup_steps    = total_steps // 20  # 5% warmup

print(f"Effective batch: {effective_batch}")
print(f"Steps/epoch: {len(train_ds) // effective_batch:,}")
print(f"Total steps: {total_steps:,}  Warmup: {warmup_steps}")
print(f"In-batch negatives per sample: {effective_batch - 1}")

model = SentenceTransformer(BASE_MODEL)
loss  = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=MICRO_BATCH * 2,  # no gradients = 2x fits
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"\n✅ Done in {elapsed:.1f} min  ({len(train_ds) * EPOCHS / elapsed:.0f} pairs/min)")

In [ ]:
# ── 6. Package model for download ───────────────────────────────────────────
# Kaggle output: /kaggle/working/ is the session output directory.
# Files written there appear in the Output tab and can be downloaded.
import os, shutil

KAGGLE_OUT = "/kaggle/working/scripture-minilm"
os.makedirs(KAGGLE_OUT, exist_ok=True)
shutil.copytree(OUT_DIR, KAGGLE_OUT, dirs_exist_ok=True)
print(f"Model saved to: {KAGGLE_OUT}")
print("Files:", [f for f in os.listdir(KAGGLE_OUT) if not f.startswith("checkpoint")])

# Zip for single-file download from the Output tab
shutil.make_archive("/kaggle/working/scripture-minilm", "zip", OUT_DIR)
zip_size = os.path.getsize("/kaggle/working/scripture-minilm.zip") // 1024 // 1024

print(f"Zip: /kaggle/working/scripture-minilm.zip  ({zip_size} MB)")print("\nDownload from: Kaggle notebook → Output tab → scripture-minilm.zip")

## After training

Download `scripture-minilm.zip` from the Kaggle notebook's **Output tab**.

On your local machine:

```bash
# 1. Download scripture-minilm.zip from Kaggle Output tab, then:
unzip scripture-minilm.zip -d resources/models/scripture-minilm/

# 2. Re-encode all 41k verses with the fine-tuned model (~3 min)
python3 scripts/rebake-embeddings.py

# 3. Re-whiten embeddings (ZCA whitening matrix has changed)
node scripts/prebake-whitening.js

# 4. Rebuild cluster labels (centroids have changed)
node scripts/prebake-cluster-labels.js

# 5. Rebuild kNN graph
node scripts/prebake-knn.js

# 6. Rebuild spectral embeddings
node scripts/prebake-spectral.js

# 7. Restart the server
npm run dev
```

### What changed from previous training

| | Round 1 | Round 3 | Round 4 (this) |
|---|---|---|---|
| Base model | `all-MiniLM-L6-v2` | `all-MiniLM-L6-v2` | `all-MiniLM-L6-v2` (same) |
| Training pairs | ~62k | ~346k | ~461k |
| Pair sources | 1 (topic→verse) | 6 | 9 |
| Key additions | — | LDS↔YLT, cross-ref, kNN | LDS↔Rotherham, Strong's lexicon |
| Effective batch | 128 | 256 | 256 |
| Epochs | 4 | 2 | 2 |
| Embedding dim | 384 | 384 | 384 |
| Est. time (T4) | ~15 min | ~15–20 min | ~20–25 min |